In [1]:
import json
import openai

In [2]:
client = openai.Client()

In [3]:
def saudacao_por_periodo(hora):
    if 5 <= 5 < 12:
        return json.dumps({"saudacao": "Bom dia!"})
    elif 12 <= hora < 18:
        return json.dumps({"saudacao": "Boa tarde!"})
    elif 18 <= hora < 22:
        return json.dumps({"saudacao": "Boa noite!"})
    else:
        return json.dumps({"saudacao": "Boa madrugada!"})

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "saudacao_por_periodo",
            "description": "Retorna uma saudação baseada na hora do dia",
            "parameters": {
                "type": "object",
                "properties":{
                    "hora":{
                        "type": "integer",
                        "description": "A hora do dia em formato de 24h"},
                     },
                    "required": ["hora"]
            }
        }
    }
]

In [5]:
funcao_disponivel = {
    "saudacao_por_periodo": saudacao_por_periodo
}

In [23]:
mensagens = [{"role": "user", "content": "Qual saudação o modelo me dá se for 20h?"}]

In [24]:
resposta = client.chat.completions.create(
    model="gpt-3.5-turbo-0125",
    messages=mensagens,
    tools=tools,
    tool_choice="auto"
)

In [25]:
mensagem_resp = resposta.choices[0].message
mensagem_resp

ChatCompletionMessage(content=None, refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_qdzLczVvkcyvtKrqDBPsjOzC', function=Function(arguments='{"hora":20}', name='saudacao_por_periodo'), type='function')])

In [26]:
tool_calls = mensagem_resp.tool_calls
tool_calls

[ChatCompletionMessageToolCall(id='call_qdzLczVvkcyvtKrqDBPsjOzC', function=Function(arguments='{"hora":20}', name='saudacao_por_periodo'), type='function')]

In [27]:
if tool_calls:
    mensagens.append(mensagem_resp)
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_to_call = funcao_disponivel[function_name]
        function_args = json.loads(tool_call.function.arguments)
        function_response = function_to_call(
            hora = function_args.get("hora")
        )
        
        mensagens.append(
            {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response
            }
        )
    
    segunda_resposta = client.chat.completions.create(
        model="gpt-3.5-turbo-0125",
        messages = mensagens
    )

In [28]:
mensagem_resposta = segunda_resposta.choices[0].message
mensagem_resposta

ChatCompletionMessage(content='Desculpe, houve um erro na identificação do período. A saudação correta para as 20h é "Boa noite!".', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)